# 00 · Environment Setup
**DS-GA 3001 · RLHF Portfolio Management**

Run this notebook once at the start of each Colab session to:
1. Mount Google Drive
2. Clone / pull the GitHub repo
3. Install all dependencies
4. Run the verification script

> **Note:** After running this notebook, use the kernel from `rlhf-portfolio/` as your working directory.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Shared project folder on Drive — adjust path if your team uses a different folder name
DRIVE_PROJECT = '/content/drive/MyDrive/WallStRL'

import os
os.makedirs(DRIVE_PROJECT, exist_ok=True)
print(f'Drive project folder: {DRIVE_PROJECT}')

## 2. Clone or pull the GitHub repo

In [ ]:
import os

REPO_URL  = 'https://github.com/YOUR_ORG/rlhf-portfolio.git'   # ← update with your repo URL
REPO_DIR  = '/content/rlhf-portfolio'

if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    print('Cloning repo...')
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

## 3. Install dependencies

In [ ]:
# Core deps from requirements.txt
!pip install -q -r requirements.txt

# FinRL — install from source (stable pip release is outdated)
!pip install -q git+https://github.com/AI4Finance-Foundation/FinRL.git

print('\nInstallation complete.')

## 4. Add src to Python path & verify

In [ ]:
import sys, os

# Make sure we're in the repo root and src is importable
repo_root = '/content/rlhf-portfolio'
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.chdir(repo_root)

# Run the verification script
!python scripts/verify_env.py

## 5. Symlink Drive folders for persistent storage

In [ ]:
import os

# Directories we want to persist across Colab sessions
PERSISTENT_DIRS = ['data', 'results/checkpoints', 'results/figures', 'runs']

for rel_path in PERSISTENT_DIRS:
    drive_path  = os.path.join(DRIVE_PROJECT, rel_path)
    local_path  = os.path.join(repo_root, rel_path)
    os.makedirs(drive_path, exist_ok=True)

    # Remove local dir and symlink to Drive
    if os.path.islink(local_path):
        print(f'  Already symlinked: {rel_path}')
    elif os.path.isdir(local_path) and not os.listdir(local_path):
        os.rmdir(local_path)
        os.symlink(drive_path, local_path)
        print(f'  Symlinked {local_path} → {drive_path}')
    else:
        print(f'  Skipped {rel_path} (non-empty or not a dir — check manually)')

print('\nDrive symlinks set up. Data and checkpoints will persist across sessions.')

## 6. Git config (first time only)

In [ ]:
# Run once — sets git identity for commits from Colab
# Each team member should fill in their own name and email
GIT_NAME  = 'Your Name'       # ← update
GIT_EMAIL = 'you@nyu.edu'     # ← update

!git config --global user.name  "{GIT_NAME}"
!git config --global user.email "{GIT_EMAIL}"
!git config --global credential.helper store

print('Git identity configured.')
print('To push, you will need a GitHub Personal Access Token (PAT).')
print('Generate one at: https://github.com/settings/tokens')

## 7. Standard commit helper

Use this cell to commit and push at the end of any work session.

In [ ]:
# ── Edit commit message then run ──────────────────────────────────────────────
COMMIT_MSG = 'WIP: describe what you changed'  # ← update before running

os.chdir(repo_root)
!git add -A
!git status --short
!git commit -m "{COMMIT_MSG}"
!git push

---
## Setup complete

You can now open any of the project notebooks:
- `notebooks/01_data.ipynb` — data download & feature engineering
- `notebooks/02_env.ipynb` — FinRL env validation  
- `notebooks/03_base_training.ipynb` — PPO base agent  
- `notebooks/04_rlhf_data.ipynb` — preference pairs + reward model training  
- `notebooks/05_finetuning.ipynb` — RLHF fine-tuning  
- `notebooks/06_evaluation.ipynb` — evaluation & plots